In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
import warnings
warnings.filterwarnings('ignore')

sns.set_style('darkgrid')

# ---------------------------
# 1. Load news data (use your actual path)
# ---------------------------
# Load news data
news = pd.read_csv('../data/raw/raw_analyst_ratings.csv')

# Extract datetime: take first 19 characters (YYYY-MM-DD HH:MM:SS)
news['date'] = news['date'].str[:19]
news['date'] = pd.to_datetime(news['date'], errors='coerce')

# Drop rows with invalid dates
news = news.dropna(subset=['date'])

print(f"Loaded {len(news)} articles with valid dates")

# ---------------------------
# 2. Sentiment analysis (TextBlob)
# ---------------------------
def get_sentiment(text):
    return TextBlob(str(text)).sentiment.polarity

# Sample first to test (remove sample for full run)
# news_sample = news.sample(10000, random_state=42)  # use this for quick test
news_sample = news  # use full dataset when ready

news_sample['sentiment'] = news_sample['headline'].apply(get_sentiment)
print("Sentiment computed")

# ---------------------------
# 3. Load stock price data for each ticker and compute daily returns
# ---------------------------
def load_stock_returns(ticker):
    df = pd.read_csv(f'../data/raw/{ticker}.csv')
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)
    df.rename(columns={'Close': 'Adj Close'}, inplace=True)
    df['daily_return'] = df['Adj Close'].pct_change() * 100  # percentage
    return df[['daily_return']].dropna()

tickers = ['AAPL', 'AMZN', 'GOOG', 'META', 'NVDA']
returns = {ticker: load_stock_returns(ticker) for ticker in tickers}

# ---------------------------
# 4. Align news dates to trading days (weekend/holiday -> next trading day)
# ---------------------------
def align_to_trading_day(date, trading_dates):
    """Find the next trading day if date is not in trading_dates"""
    if date in trading_dates:
        return date
    # Find next trading day
    future = trading_dates[trading_dates >= date]
    if len(future) > 0:
        return future.min()
    return None

# Prepare results
all_correlations = []

for ticker in tickers:
    # Get news for this stock
    news_ticker = news_sample[news_sample['stock'] == ticker].copy()
    if len(news_ticker) == 0:
        print(f"No news for {ticker}")
        continue
    
    # Get trading dates for this stock
    trading_dates = returns[ticker].index
    
    # Align each news date to trading day
    news_ticker['trading_date'] = news_ticker['date'].apply(
        lambda d: align_to_trading_day(d, trading_dates)
    )
    news_ticker = news_ticker.dropna(subset=['trading_date'])
    
    # Aggregate sentiment per trading day
    daily_sentiment = news_ticker.groupby('trading_date')['sentiment'].mean()
    
    # Merge with returns
    merged = pd.DataFrame(daily_sentiment).join(returns[ticker], how='inner')
    
    if len(merged) < 5:
        print(f"Not enough aligned data for {ticker}")
        continue
    
    # Calculate Pearson correlation
    corr = merged['sentiment'].corr(merged['daily_return'])
    all_correlations.append({'ticker': ticker, 'correlation': corr, 'data': merged})
    
    print(f"{ticker}: correlation = {corr:.4f} (based on {len(merged)} days)")

# ---------------------------
# 5. Visualization for each stock
# ---------------------------
for item in all_correlations:
    ticker = item['ticker']
    merged = item['data']
    corr = item['correlation']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Scatter plot
    axes[0].scatter(merged['sentiment'], merged['daily_return'], alpha=0.5)
    axes[0].axhline(0, color='gray', linestyle='--')
    axes[0].axvline(0, color='gray', linestyle='--')
    axes[0].set_title(f'{ticker} - Sentiment vs Daily Return (r = {corr:.3f})')
    axes[0].set_xlabel('Sentiment Polarity')
    axes[0].set_ylabel('Daily Return (%)')
    
    # Bar chart by sentiment category
    def categorize(score):
        if score > 0.05:
            return 'Positive'
        elif score < -0.05:
            return 'Negative'
        else:
            return 'Neutral'
    
    merged['category'] = merged['sentiment'].apply(categorize)
    avg_return = merged.groupby('category')['daily_return'].mean().reindex(['Positive', 'Neutral', 'Negative'])
    
    axes[1].bar(avg_return.index, avg_return.values, color=['green', 'gray', 'red'])
    axes[1].set_title(f'{ticker} - Avg Daily Return by Sentiment')
    axes[1].set_ylabel('Avg Daily Return (%)')
    
    plt.tight_layout()
    plt.show()

# ---------------------------
# 6. Summary table
# ---------------------------
summary = pd.DataFrame([{'Stock': item['ticker'], 'Correlation': item['correlation']} for item in all_correlations])
print("\n=== Correlation Summary ===")
print(summary)

ValueError: time data "2020-05-22 00:00:00" doesn't match format "%Y-%m-%d %H:%M:%S%z". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "textblob"])
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\beti\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True